# SpeedPilot AI - Free Model Fine-Tuning with Unsloth
Train your own custom PageSpeed & Core Web Vitals remediation AI for **100% FREE** using open-weights models on Google Colab's free T4 GPU.

Powered by:
- **Framework**: [Unsloth](https://github.com/unslothai/unsloth) (5x faster, 80% less memory)
- **Base Model**: `unsloth/Qwen2.5-Coder-7B-Instruct` or `unsloth/Llama-3.2-3B-Instruct`
- **Dataset**: 60+ Gold-Standard Multi-Persona CWV Remediations from `@speedpilot/ai-engine`

In [ ]:
# 1. Install Unsloth and training dependencies (Run on free GPU runtime)
!pip install --no-deps "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers<0.0.27" trl peft accelerate bitsandbytes datasets

In [ ]:
# 2. Load the 4-bit quantized base model
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
model_name = "unsloth/Qwen2.5-Coder-7B-Instruct"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=max_seq_length,
    dtype=None,
    load_in_4bit=True,
)

# Add LoRA adapters
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)
print("Model initialized with 4-bit LoRA adapters successfully!")

In [ ]:
# 3. Upload or load speedpilot-finetune.jsonl
import os
from datasets import load_dataset

# If running in Colab and file is not yet present, prompt upload widget:
if not os.path.exists("speedpilot-finetune.jsonl"):
    try:
        from google.colab import files
        print("Upload speedpilot-finetune.jsonl from your computer:")
        uploaded = files.upload()
    except ImportError:
        pass

dataset = load_dataset("json", data_files="speedpilot-finetune.jsonl", split="train")
print(f"Loaded {len(dataset)} SpeedPilot training pairs successfully!")

# Format prompts with Qwen/Llama chat template
def format_prompts(batch):
    formatted = []
    for messages in batch["messages"]:
        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
        formatted.append(text)
    return {"text": formatted}

dataset = dataset.map(format_prompts, batched=True)

In [ ]:
# 4. Train the model using Hugging Face TRL SFTTrainer
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        max_steps=80,
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=5,
        output_dir="speedpilot-outputs",
        optim="adamw_8bit",
        seed=3407,
    ),
)

trainer_stats = trainer.train()
print("SpeedPilot Fine-Tuning Finished!")

In [ ]:
# 5. Test Inference with a real PageSpeed issue prompt
FastLanguageModel.for_inference(model)

test_messages = [
    {
        "role": "system",
        "content": "You are SpeedPilot AI, an elite Web Performance Engineering AI that writes concrete, copy-pasteable code fixes for PageSpeed and Core Web Vitals audits across multiple stakeholder personas."
    },
    {
        "role": "user",
        "content": "Target: https://my-store.com\nTech Stack: SHOPIFY\nIssue: render-blocking-resources\nFlagged: https://my-store.com/cdn/shop/t/1/assets/theme.css\nProvide exact multi-persona remediation with real code."
    }
]

inputs = tokenizer.apply_chat_template(
    test_messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt",
).to("cuda")

outputs = model.generate(input_ids=inputs, max_new_tokens=512, use_cache=True)
print(tokenizer.batch_decode(outputs[:, inputs.shape[1]:], skip_special_tokens=True)[0])

In [ ]:
# 6. Export GGUF for local Ollama usage and download
import glob
model.save_pretrained_gguf("speedpilot-ai-gguf", tokenizer, quantization_method="q4_k_m")
print("Exported speedpilot-ai-gguf/unsloth.Q4_K_M.gguf!")

# Trigger browser download in Google Colab:
try:
    from google.colab import files
    gguf_files = glob.glob("speedpilot-ai-gguf/*.gguf")
    if gguf_files:
        print(f"Downloading {gguf_files[0]} to your local computer...")
        files.download(gguf_files[0])
except ImportError:
    pass